# LO benchmark results

Load multi-model results for the linear-optimization (LO / LP) benchmark into one dataframe, then show a **score table**: rows = questions, columns = LLMs.

Each cell is `1` if the model's JSON `cost` was within 1% of the keyed objective, else `0`.

**Download in this notebook:** the load cell sets `DOWNLOAD_KAGGLE_RUNS = True` and pulls runs via the Kaggle CLI into `data/kaggle_runs/lo-normative-accuracy-3`. Requires `kaggle auth login` once.

If download is off and files are missing, the cell will still try to download automatically.

Or from a local sandbox / merged CSV: set `LOAD_FROM_KAGGLE = False` and put `*.run.json` or `*merged*.csv` under `data/lp/` or one of the sandbox candidate folders.

In [1]:
! \src\projects\sceptical_llms\.venv\Scripts\python.exe -m kaggle auth login

The system cannot find the path specified.


In [2]:
# Ensure dependencies for whatever kernel Cursor/VS Code selected.
import importlib.util
import subprocess
import sys

for pkg in ("pandas", "kaggle"):
    if importlib.util.find_spec(pkg) is None:
        print(f"Installing {pkg} into: {sys.executable}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
    else:
        print(f"{pkg} available in: {sys.executable}")


pandas available in: c:\src2\Nemotron\.venv-examine\Scripts\python.exe
kaggle available in: c:\src2\Nemotron\.venv-examine\Scripts\python.exe


In [3]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "lp").is_dir():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import benchmarks.kaggle_runs as kaggle_runs
import benchmarks.lp_rate as lp_rate

importlib.reload(lp_rate)
importlib.reload(kaggle_runs)

from benchmarks.kaggle_runs import (
    DEFAULT_LO_TASK_SLUG,
    download_task_runs,
    load_base_rate_run_rows_from_tree,
    merged_lo_results_from_kaggle_runs,
)
from benchmarks.lp_rate import write_merged_results_csv

# --- knobs ---
LOAD_FROM_KAGGLE = True
DOWNLOAD_KAGGLE_RUNS = True  # download via Kaggle CLI into data/kaggle_runs/...
# Task published by %choose lo_normative_accuracy_3
KAGGLE_TASK_SLUG = DEFAULT_LO_TASK_SLUG  # "lo-normative-accuracy-3"
assert KAGGLE_TASK_SLUG == "lo-normative-accuracy-3", KAGGLE_TASK_SLUG

# Sandbox / alternate run folders (first existing wins when LOAD_FROM_KAGGLE).
SANDBOX_CANDIDATES = [
    ROOT / "data" / "kaggle_runs" / "lo-normative-accuracy-3",
    Path("/kaggle/working") / "lp_benchmark",
    Path("/kaggle/working"),
]

BENCHMARK_CSV = ROOT / "data" / "lp" / "benchmark.csv"
MERGED_DIR = ROOT / "data" / "lp"


def _first_existing(paths: list[Path]) -> Path | None:
    for path in paths:
        if path.is_dir() and (
            any(path.rglob("*.run.json")) or any(path.glob("*merged*.csv"))
        ):
            return path
    return None


def _load_merged_from_run_tree(runs_dir: Path) -> tuple[pd.DataFrame, str]:
    try:
        merged_rows = merged_lo_results_from_kaggle_runs(
            runs_dir,
            benchmark_path=BENCHMARK_CSV,
            fill_missing=False,
        )
    except ValueError:
        # Fallback for sandbox trees that do not match the task-slug filter path.
        run_rows = load_base_rate_run_rows_from_tree(runs_dir)
        if not run_rows:
            raise
        out_csv = MERGED_DIR / "lo_merged_results.csv"
        write_merged_results_csv(
            run_rows,
            out_csv,
            benchmark_path=BENCHMARK_CSV,
        )
        merged_rows = pd.read_csv(out_csv).to_dict(orient="records")
    return pd.DataFrame(merged_rows), f"run tree ({runs_dir})"


if LOAD_FROM_KAGGLE:
    default_runs = ROOT / "data" / "kaggle_runs" / KAGGLE_TASK_SLUG
    if DOWNLOAD_KAGGLE_RUNS:
        print(f"Downloading {KAGGLE_TASK_SLUG} -> {default_runs}")
        default_runs.mkdir(parents=True, exist_ok=True)
        download_task_runs(KAGGLE_TASK_SLUG, default_runs)

    # Prefer the _2 download dir; do not fall back to lo-normative-accuracy (v1).
    runs_dir = _first_existing([default_runs, *SANDBOX_CANDIDATES])
    if runs_dir is None and not DOWNLOAD_KAGGLE_RUNS:
        print(f"No local runs found; downloading {KAGGLE_TASK_SLUG} -> {default_runs}")
        default_runs.mkdir(parents=True, exist_ok=True)
        download_task_runs(KAGGLE_TASK_SLUG, default_runs)
        runs_dir = _first_existing([default_runs, *SANDBOX_CANDIDATES])

    if runs_dir is None:
        raise FileNotFoundError(
            "No LO run results found after download attempt. Looked under:\n  - "
            + "\n  - ".join(str(p) for p in [default_runs, *SANDBOX_CANDIDATES])
            + f"\nManual download:\n  python -m kaggle benchmarks tasks download "
            f"{KAGGLE_TASK_SLUG} -o {default_runs}"
        )

    if any(runs_dir.rglob("*.run.json")):
        df, data_source = _load_merged_from_run_tree(runs_dir)
    else:
        merged_csv = sorted(
            runs_dir.glob("*merged*.csv"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )[0]
        df = pd.read_csv(merged_csv)
        data_source = str(merged_csv)
else:
    merged_candidates = sorted(
        list(MERGED_DIR.glob("*merged*.csv"))
        + list(MERGED_DIR.glob("lo_merged_results*.csv")),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    # De-dupe while preserving mtime order.
    seen: set[Path] = set()
    unique: list[Path] = []
    for path in merged_candidates:
        if path not in seen:
            seen.add(path)
            unique.append(path)
    if not unique:
        raise FileNotFoundError(
            f"Missing merged results under {MERGED_DIR}. "
            "Set LOAD_FROM_KAGGLE=True or run benchmark/lo-benchmark.ipynb."
        )
    merged_csv = unique[0]
    df = pd.read_csv(merged_csv)
    data_source = str(merged_csv)

if "score" not in df.columns:
    raise KeyError("Merged data must include 'score'.")

df["score_value"] = df["score"].astype(str).str.lower().eq("true").astype(int)
if "naive_lp_confusion" in df.columns:
    df["naive_value"] = (
        df["naive_lp_confusion"].astype(str).str.lower().eq("true").astype(int)
    )

print("Source:", data_source)
print("Task slug:", KAGGLE_TASK_SLUG)
print("Rows:", len(df))
print("Models:", sorted(df["model"].dropna().unique()))
print("Questions:", sorted(df["example_id"].unique()) if "example_id" in df else "?")
df.head()


Skipped 14 stale run row(s) with example_ids not in benchmark.csv.
Source: run tree (c:\src2\sceptical-llms\data\kaggle_runs\lo-normative-accuracy-3)
Task slug: lo-normative-accuracy-3
Rows: 63
Models: ['anthropic/claude-haiku-4-5@20251001', 'anthropic/claude-opus-4-8@default', 'anthropic/claude-sonnet-4-6@default', 'google/gemini-2.5-flash', 'google/gemini-3-flash-preview', 'google/gemini-3.5-flash', 'openai/gpt-5.6-terra']
Questions: ['charter_buses__integrality__explicit__json', 'charter_buses__integrality__json', 'fund_allocation__nonnegativity__explicit__json', 'fund_allocation__nonnegativity__json', 'gift_baskets__both__explicit__json', 'gift_baskets__both__json', 'warehouse_shipping__nonnegativity__explicit__json', 'warehouse_shipping__nonnegativity__json', 'workshop_vehicles__none__json']


,example_id,vignette_name,failure_mode,condition,problem_type,intersection_size,response_type,has_statistics,variant,prompt,...,parsed_confidence,comment_line,scoring_type,parseable,score,naive_lp_confusion,parsed_objective,parsed_solution,score_value,naive_value
0,charter_buses__integrality__explicit__json,charter buses,integrality,explicit,explicit_constraints,,json,true,json,You are an operations consultant. Your task is...,...,,,open,true,false,false,2461.5,"{""large_buses"": 3, ""small_buses"": 0}",0,0
1,charter_buses__integrality__explicit__json,charter buses,integrality,explicit,explicit_constraints,,json,true,json,You are an operations consultant. Your task is...,...,,,open,true,true,false,2341.25,"{""large_buses"": 2, ""small_buses"": 1}",1,0
2,charter_buses__integrality__explicit__json,charter buses,integrality,explicit,explicit_constraints,,json,true,json,You are an operations consultant. Your task is...,...,,,open,true,true,false,2341.25,"{""large_buses"": 2, ""small_buses"": 1}",1,0
3,charter_buses__integrality__explicit__json,charter buses,integrality,explicit,explicit_constraints,,json,true,json,You are an operations consultant. Your task is...,...,,,open,true,false,false,2461.5,"{""large_buses"": 3, ""small_buses"": 0}",0,0
4,charter_buses__integrality__explicit__json,charter buses,integrality,explicit,explicit_constraints,,json,true,json,You are an operations consultant. Your task is...,...,,,open,true,true,false,2341.25,"{""large_buses"": 2, ""small_buses"": 1}",1,0


## Score table: vignette × version × LLM

Rows are LO vignettes with separate versions (`condition`: implicit / explicit / control). Columns are models. Cell = mean keyed score (`1` = correct within 1%, `0` = incorrect).

In [4]:
plot_df = df.copy()
if "condition" in plot_df.columns and "vignette_name" in plot_df.columns:
    index_cols: list[str] | str = ["vignette_name", "condition"]
elif "vignette_name" in plot_df.columns:
    index_cols = "vignette_name"
else:
    index_cols = "example_id"

# Stable version order within each vignette.
if isinstance(index_cols, list) and "condition" in index_cols:
    condition_order = {"implicit": 0, "explicit": 1, "control": 2}
    plot_df = plot_df.assign(
        _condition_ord=plot_df["condition"].map(condition_order).fillna(99)
    ).sort_values(["vignette_name", "_condition_ord"])

score_table = (
    plot_df.pivot_table(
        index=index_cols,
        columns="model",
        values="score_value",
        aggfunc="mean",
    )
    .sort_index(axis=1)
)

# Keep vignette/version row order; append an overall mean row.
if isinstance(index_cols, list):
    # Reindex rows to sorted vignette then condition order.
    ordered = plot_df[["vignette_name", "condition"]].drop_duplicates()
    score_table = score_table.reindex(
        pd.MultiIndex.from_frame(ordered, names=["vignette_name", "condition"])
    )
else:
    score_table = score_table.sort_index()

# Column means first (before adding the row-mean column), then append mean row
# with .values so MultiIndex loc does not mis-align into NaNs.
col_means = score_table.mean(axis=0)
score_table["mean"] = score_table.mean(axis=1)
mean_row = col_means.copy()
mean_row["mean"] = float(col_means.mean())
if isinstance(score_table.index, pd.MultiIndex):
    score_table.loc[("mean", ""), :] = mean_row.to_numpy()
else:
    score_table.loc["mean", :] = mean_row.to_numpy()

display_table = score_table.round(3)
display_table


model                         anthropic/claude-haiku-4-5@20251001  \
vignette_name      condition                                        
charter buses      implicit                                 1.000   
                   explicit                                 0.000   
fund allocation    implicit                                 0.000   
                   explicit                                 0.000   
gift baskets       implicit                                 0.000   
                   explicit                                 0.000   
warehouse shipping implicit                                 0.000   
                   explicit                                 0.000   
workshop vehicles  control                                  0.000   
mean                                                        0.111   

model                         anthropic/claude-opus-4-8@default  \
vignette_name      condition                                      
charter buses      implicit                               1.000   
                   explicit                               1.000   
fund allocation    implicit                               0.000   
                   explicit                               0.000   
gift baskets       implicit                               0.000   
                   explicit                               0.000   
warehouse shipping implicit                               0.000   
                   explicit                               0.000   
workshop vehicles  control                                0.000   
mean                                                      0.222   

model                         anthropic/claude-sonnet-4-6@default  \
vignette_name      condition                                        
charter buses      implicit                                 0.000   
                   explicit                                 1.000   
fund allocation    implicit                                 0.000   
                   explicit                                 0.000   
gift baskets       implicit                                 1.000   
                   explicit                                 1.000   
warehouse shipping implicit                                 0.000   
                   explicit                                 0.000   
workshop vehicles  control                                  1.000   
mean                                                        0.444   

model                         google/gemini-2.5-flash  \
vignette_name      condition                            
charter buses      implicit                       0.0   
                   explicit                       0.0   
fund allocation    implicit                       0.0   
                   explicit                       0.0   
gift baskets       implicit                       0.0   
                   explicit                       0.0   
warehouse shipping implicit                       0.0   
                   explicit                       0.0   
workshop vehicles  control                        0.0   
mean                                              0.0   

model                         google/gemini-3-flash-preview  \
vignette_name      condition                                  
charter buses      implicit                           1.000   
                   explicit                           1.000   
fund allocation    implicit                           0.000   
                   explicit                           0.000   
gift baskets       implicit                           1.000   
                   explicit                           1.000   
warehouse shipping implicit                           0.000   
                   explicit                           0.000   
workshop vehicles  control                            1.000   
mean                                                  0.556   

model                         google/gemini-3.5-flash  openai/gpt-5.6-terra  \
vignette_name   

## Optional: naive-LP confusion (vignette × version × LLM)

`1` means the model reported the stated-constraints-only optimum (within 1%). Same row layout as the score table (vignette × condition).

In [5]:
if "naive_value" in df.columns:
    naive_df = df.copy()
    if "condition" in naive_df.columns and "vignette_name" in naive_df.columns:
        naive_index: list[str] | str = ["vignette_name", "condition"]
        condition_order = {"implicit": 0, "explicit": 1, "control": 2}
        naive_df = naive_df.assign(
            _condition_ord=naive_df["condition"].map(condition_order).fillna(99)
        ).sort_values(["vignette_name", "_condition_ord"])
    elif "vignette_name" in naive_df.columns:
        naive_index = "vignette_name"
    else:
        naive_index = "example_id"
    naive_table = (
        naive_df.pivot_table(
            index=naive_index,
            columns="model",
            values="naive_value",
            aggfunc="mean",
        )
        .sort_index(axis=1)
    )
    if isinstance(naive_index, list):
        ordered = naive_df[["vignette_name", "condition"]].drop_duplicates()
        naive_table = naive_table.reindex(
            pd.MultiIndex.from_frame(ordered, names=["vignette_name", "condition"])
        )
    else:
        naive_table = naive_table.sort_index()
    naive_table["mean"] = naive_table.mean(axis=1)
    # Avoid MultiIndex loc Series-alignment NaNs on the summary row.
    col_means = naive_table.drop(columns=["mean"]).mean(axis=0)
    mean_row = col_means.copy()
    mean_row["mean"] = float(col_means.mean())
    if isinstance(naive_table.index, pd.MultiIndex):
        naive_table.loc[("mean", ""), :] = mean_row.to_numpy()
    else:
        naive_table.loc["mean", :] = mean_row.to_numpy()
    display(naive_table.round(3))
else:
    print("No naive_lp_confusion column in merged results.")


model                         anthropic/claude-haiku-4-5@20251001  \
vignette_name      condition                                        
charter buses      implicit                                   0.0   
                   explicit                                   0.0   
fund allocation    implicit                                   0.0   
                   explicit                                   0.0   
gift baskets       implicit                                   0.0   
                   explicit                                   0.0   
warehouse shipping implicit                                   0.0   
                   explicit                                   0.0   
workshop vehicles  control                                    0.0   
mean                                                          0.0   

model                         anthropic/claude-opus-4-8@default  \
vignette_name      condition                                      
charter buses      implicit                                 0.0   
                   explicit                                 0.0   
fund allocation    implicit                                 0.0   
                   explicit                                 0.0   
gift baskets       implicit                                 0.0   
                   explicit                                 0.0   
warehouse shipping implicit                                 0.0   
                   explicit                                 0.0   
workshop vehicles  control                                  0.0   
mean                                                        0.0   

model                         anthropic/claude-sonnet-4-6@default  \
vignette_name      condition                                        
charter buses      implicit                                   0.0   
                   explicit                                   0.0   
fund allocation    implicit                                   0.0   
                   explicit                                   0.0   
gift baskets       implicit                                   0.0   
                   explicit                                   0.0   
warehouse shipping implicit                                   0.0   
                   explicit                                   0.0   
workshop vehicles  control                                    0.0   
mean                                                          0.0   

model                         google/gemini-2.5-flash  \
vignette_name      condition                            
charter buses      implicit                       0.0   
                   explicit                       0.0   
fund allocation    implicit                       0.0   
                   explicit                       0.0   
gift baskets       implicit                       0.0   
                   explicit                       0.0   
warehouse shipping implicit                       0.0   
                   explicit                       0.0   
workshop vehicles  control                        0.0   
mean                                              0.0   

model                         google/gemini-3-flash-preview  \
vignette_name      condition                                  
charter buses      implicit                             0.0   
                   explicit                             0.0   
fund allocation    implicit                             0.0   
                   explicit                             0.0   
gift baskets       implicit                             0.0   
                   explicit                             0.0   
warehouse shipping implicit                             0.0   
                   explicit                             0.0   
workshop vehicles  control                              0.0   
mean                                                    0.0   

model                         google/gemini-3.5-flash  openai/gpt-5.6-terra  \
vignette_name   

In [6]:
# Persist the main score table next to the LP data.
out_path = MERGED_DIR / "lo_score_by_question_model.csv"
score_table.to_csv(out_path)
print("Wrote", out_path)

Wrote c:\src2\sceptical-llms\data\lp\lo_score_by_question_model.csv
